# AutoPBR v2 — Run Notebook (molto commentato)

Notebook orchestratore per il repo `dera8/AutoPBR`.

## Cosa fa
1. (Opzionale) Scarica/organizza dati e foto (`1_download_data.py`)
2. Segmenta le foto in **road / wall / roof** con SAM3 (`2_sam3hi.py`)
3. Esegue un **sanity-check** con proxy semantic segmentation (`3_proxy_semantic_check.py`)
4. Esegue Nemotron in modalità **structure-first** con priors per-building (`4_nemotron_building_priors.py`)
5. (Opzionale) Baseline **full-image** (unstructured) (`5_nemotron_fullimage.py`)

## Perché eseguiamo via CLI (non import Python)
- stesso comportamento del terminale
- log live
- riproducibilità (anche per reviewer)

Assunzioni:
- script nella **root** del repo
- immagini input in `photos2/`
- `NVIDIA_API_KEY` impostata per Nemotron


## 0) CONFIG — controlla qui una volta sola


In [ ]:
from pathlib import Path
import os, sys, subprocess, shlex

REPO_ROOT = Path(".").resolve()
SCRIPTS_DIR = REPO_ROOT          # se sposti gli script in scripts/, cambia qui
PHOTOS_DIR = REPO_ROOT / "photos2"

OUT_STRUCTURED_JSON = REPO_ROOT / "materials_hybrid.json"
OUT_BASELINE_JSON   = REPO_ROOT / "baseline_full_image.json"

print("Repo:", REPO_ROOT)
print("Scripts:", SCRIPTS_DIR)
print("Photos:", PHOTOS_DIR)


## 0.1) API Key — dove metterla

### Opzione A (consigliata): da terminale prima di aprire Jupyter
Linux/Mac:
```bash
export NVIDIA_API_KEY="..."
jupyter notebook
```

Windows PowerShell:
```powershell
setx NVIDIA_API_KEY "..."
```
poi riapri il terminale.

### Opzione B: temporaneamente qui (vale solo finché il kernel è acceso)
Scommenta e inserisci la chiave nella prossima cella.

> ⚠️ Non committare mai la chiave nel repo.


In [ ]:
# os.environ["NVIDIA_API_KEY"] = "..."  # <-- scommenta solo se vuoi usarla temporaneamente qui
print("NVIDIA_API_KEY present?", bool(os.environ.get("NVIDIA_API_KEY")))


## 0.2) Helper — esecuzione comandi con output live


In [ ]:
def run(cmd: str):
    # Esegue un comando come in terminale e mostra output live.
    # Se il comando ritorna errore, questa cella fallisce (così sai subito dove).
    print("\n▶", cmd)
    subprocess.run(shlex.split(cmd), check=True)

def py(script: str, args: str = ""):
    # Esegue uno script Python del repo usando l'interprete corrente del notebook.
    script_path = (SCRIPTS_DIR / script).resolve()
    if not script_path.exists():
        raise FileNotFoundError(f"Script not found: {script_path}")
    run(f"{sys.executable} {script_path} {args}".strip())


## 0.3) Sanity checks — prima di sprecare tempo

- Crea `photos2/` se non esiste
- Conta immagini disponibili
- Controlla che `NVIDIA_API_KEY` sia impostata (necessaria per Nemotron)


In [ ]:
PHOTOS_DIR.mkdir(exist_ok=True, parents=True)

imgs = (
    list(PHOTOS_DIR.glob("*.jpg")) +
    list(PHOTOS_DIR.glob("*.jpeg")) +
    list(PHOTOS_DIR.glob("*.png"))
)
print(f"📷 Immagini trovate in {PHOTOS_DIR}: {len(imgs)}")
if len(imgs) == 0:
    print("⚠️ Metti le immagini dentro photos2/ prima di Run All.")

if not os.environ.get("NVIDIA_API_KEY"):
    print("⚠️ NVIDIA_API_KEY non impostata: gli step 4/5 (Nemotron) falliranno.")
else:
    print("✅ NVIDIA_API_KEY OK")


# 1) (Opzionale) Download data — `1_download_data.py`

Usalo quando vuoi creare/aggiornare:
- `photos_manifest.json`
- `map.osm`
- `buildings.geojson`

Se hai già questi file e stai lavorando su un set fisso di foto, puoi saltare.

Esempio:
```python
py("1_download_data.py", '--bbox "minLon,minLat,maxLon,maxLat" --profile autopbr')
```


In [ ]:
# py("1_download_data.py", '--bbox "5.32045,60.39670,5.32650,60.39990" --profile autopbr')
print("Step 1 è opzionale: scommenta e aggiusta args se ti serve.")


# 2) Semantic segmentation (SAM3) — `2_sam3hi.py`

Segmenta ogni immagine in road/wall/roof (e salva output intermedi).
Consiglio: testa prima con poche immagini (3–5) per verificare che tutto funzioni.


In [ ]:
py("2_sam3hi.py")


# 3) Proxy semantic check — `3_proxy_semantic_check.py`

Sanity-check: confronta i risultati SAM3 con proxy semantic models.
Non è ground truth, ma aiuta a scovare errori grossi e produce overlay utili per debug/appendix.


In [ ]:
py("3_proxy_semantic_check.py")


# 4) Nemotron structured (building priors) — `4_nemotron_building_priors.py`

Cuore AutoPBR: structure-first → masked VLM → JSON strutturato.

Se fallisce:
- controlla `NVIDIA_API_KEY`
- verifica che gli step 2/3 abbiano prodotto gli output attesi
- prova con poche immagini per escludere rate limit


In [ ]:
py("4_nemotron_building_priors.py")


# 5) (Opzionale) Baseline full-image — `5_nemotron_fullimage.py`

Baseline unstructured: Nemotron vede l’immagine intera.
Utile per ablation: structured vs unstructured.


In [ ]:
# py("5_nemotron_fullimage.py")
print("Step 5 opzionale: scommenta per eseguire la baseline full-image.")


# 6) Quick output check

Controlla che alcuni file chiave esistano.
Se i tuoi script usano nomi diversi, modifica la lista `expected`.


In [ ]:
expected = [
    OUT_STRUCTURED_JSON,
    OUT_BASELINE_JSON,
    REPO_ROOT / "materials_per_image_v2.json",
    REPO_ROOT / "materials_v2_filtered.json",
    REPO_ROOT / "photos_manifest.json",
]

for p in expected:
    print(("✅" if p.exists() else "❌"), p.name)

all_files = [p for p in REPO_ROOT.rglob("*") if p.is_file() and ".git" not in str(p)]
all_files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
print("\n🕒 Ultimi 10 file aggiornati:")
for p in all_files[:10]:
    print("-", p.relative_to(REPO_ROOT))
